In [9]:
import cv2
import os

# ===================== Paths =====================
image_path = r"D:\Papers\Braille\Dataset\train\images\0000007_jpg.rf.f2756c5acda9d8e2df6ddba87d051f18.jpg"
label_path = image_path.replace("images", "labels").replace(".jpg", ".txt")

# Load class names (same order as your YOLO dataset)
names = [
    "A","B","C","D","E","F","G","H","I","J",
    "K","L","M","N","O","P","Q","R","S","T",
    "U","V","W","X","Y","Z"
]  # <-- replace with your actual model.names if different

# ===================== Load Image =====================
img = cv2.imread(image_path)
h, w, _ = img.shape

# ===================== Read Labels =====================
if not os.path.exists(label_path):
    print("⚠️ Label file not found:", label_path)
    exit()

with open(label_path, "r") as f:
    labels = f.readlines()

for line in labels:
    cls, xc, yc, bw, bh = map(float, line.strip().split())
    cls = int(cls)

    # Convert normalized YOLO → pixel coordinates
    xc, yc, bw, bh = xc * w, yc * h, bw * w, bh * h
    x1, y1 = int(xc - bw / 2), int(yc - bh / 2)
    x2, y2 = int(xc + bw / 2), int(yc + bh / 2)

    # Draw bounding box
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # Draw label text
    label_text = names[cls] if cls < len(names) else str(cls)
    cv2.putText(img, label_text, (x1, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

# ===================== Show Result =====================
cv2.imshow("Annotations", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Optional: Save for checking later
cv2.imwrite("debug_annotations.jpg", img)
print("✅ Saved annotated image as debug_annotations.jpg")


UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 57: character maps to <undefined>

In [1]:
import os
import cv2
import numpy as np
from ultralytics import YOLO
from datetime import datetime

# ===================== Load YOLO model =====================
model = YOLO(r"D:\Papers\Braille\braille_char\v8n\weights\best.pt")

# ===================== Snap detections to grid =====================
def snap_to_grid(img, detections, cell_w=40, cell_h=60):
    """
    Snap YOLO detections into a uniform 2x3 Braille grid.
    Fill missing cells as 'blank' (space).
    """
    h, w = img.shape[:2]
    grid = []

    # Estimate number of cells per row/col
    cols = w // cell_w
    rows = h // cell_h

    for r in range(rows):
        row_chars = []
        for c in range(cols):
            x1, y1 = c * cell_w, r * cell_h
            x2, y2 = x1 + cell_w, y1 + cell_h

            # Match detection inside this cell
            match = None
            for (box, label) in detections:
                bx1, by1, bx2, by2 = box
                if bx1 >= x1 and by1 >= y1 and bx2 <= x2 and by2 <= y2:
                    match = label
                    break

            if match is None:
                match = " "  # blank

            row_chars.append(match)
        grid.append(row_chars)

    return grid

# ===================== Save Grid to Text =====================
def save_grid_as_text(grid, out_file):
    with open(out_file, "w", encoding="utf-8") as f:
        for row in grid:
            f.write("".join(row) + "\n")
    print(f"✅ Braille grid saved to: {out_file}")

# ===================== Main Processing =====================
def process_image_with_yolo_grid(model, image_path, output_dir):
    # Create output directories
    os.makedirs(output_dir, exist_ok=True)

    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ Error: Could not load image {image_path}")
        return

    # Run YOLO
    results = model.predict(
        img,
        conf=0.2,
        iou=0.6,
        agnostic_nms=True,
        max_det=200,
        verbose=False
    )
    r0 = results[0]

    # Extract detections
    boxes = r0.boxes.xyxy.cpu().numpy() if r0.boxes is not None else []
    cls_ids = r0.boxes.cls.cpu().numpy().astype(int) if r0.boxes is not None else []
    detections = []

    for box, cls_id in zip(boxes, cls_ids):
        label = r0.names[cls_id] if hasattr(r0, "names") else str(cls_id)
        detections.append((box, label))

    # Snap into Braille grid
    grid = snap_to_grid(img, detections)

    # Save grid to text file
    out_txt = os.path.join(output_dir, "braille_grid.txt")
    save_grid_as_text(grid, out_txt)

    # Save YOLO annotated image
    annotated_img = r0.plot()
    annotated_path = os.path.join(output_dir, "annotated_image.png")
    cv2.imwrite(annotated_path, annotated_img)
    print(f"✅ Saved annotated image: {annotated_path}")

# ===================== Run =====================
if __name__ == "__main__":
    image_path = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\Screenshot 2025-09-05 160624.png"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"yolo_grid_detection_{timestamp}"

    process_image_with_yolo_grid(model, image_path, output_dir)
    print(f"🎯 Processing complete. Results in: {output_dir}")


✅ Braille grid saved to: yolo_grid_detection_20250909_114920\braille_grid.txt
✅ Saved annotated image: yolo_grid_detection_20250909_114920\annotated_image.png
🎯 Processing complete. Results in: yolo_grid_detection_20250909_114920


In [ ]:
import cv2
import torch
from ultralytics import YOLO
import numpy as np

# ===================== Config =====================
MODEL_PATH = r"D:\Papers\Braille\braille_char\v8n\weights\best.pt"
IMAGE_PATH = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\Screenshot 2025-09-05 160624.png"

CONF_THRESH = 0.2
NMS_IOU = 0.05  # Very low to allow overlapping dots
SOFTNMS_SIGMA = 0.5


# ===================== IoU Function =====================
def bbox_iou(box1, box2):
    """
    box1: (N,4), box2: (M,4)
    xyxy format: [x1, y1, x2, y2]
    """
    x1 = torch.max(box1[:, None, 0], box2[:, 0])
    y1 = torch.max(box1[:, None, 1], box2[:, 1])
    x2 = torch.min(box1[:, None, 2], box2[:, 2])
    y2 = torch.min(box1[:, None, 3], box2[:, 3])

    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])

    union = area1[:, None] + area2 - inter
    return inter / (union + 1e-6)


# ===================== Soft-NMS =====================
def soft_nms(boxes, scores, iou_thresh=0.05, sigma=0.5, score_thresh=0.001):
    boxes = boxes.clone()
    scores = scores.clone()
    N = boxes.shape[0]

    for i in range(N):
        max_pos = i + torch.argmax(scores[i:])
        # Swap highest score box to position i
        boxes[i], boxes[max_pos] = boxes[max_pos].clone(), boxes[i].clone()
        scores[i], scores[max_pos] = scores[max_pos], scores[i]

        if i < N - 1:
            ious = bbox_iou(boxes[i].unsqueeze(0), boxes[i + 1:]).squeeze(0)
            decay = torch.exp(-(ious ** 2) / sigma)
            scores[i + 1:] *= decay

    keep = scores > score_thresh
    return boxes[keep], scores[keep]


# ===================== Optional tiling =====================
def create_tiles(img, tile_size=(640, 640), overlap=0.2):
    h, w, _ = img.shape
    th, tw = tile_size
    step_h = int(th * (1 - overlap))
    step_w = int(tw * (1 - overlap))
    tiles = []
    coords = []
    for y in range(0, h, step_h):
        for x in range(0, w, step_w):
            y1 = y
            x1 = x
            y2 = min(y + th, h)
            x2 = min(x + tw, w)
            tiles.append(img[y1:y2, x1:x2])
            coords.append((x1, y1))
    return tiles, coords


# ===================== Main Detection =====================
def detect_braille(model_path, image_path, use_tiling=True):
    model = YOLO(model_path)
    img = cv2.imread(image_path)

    all_boxes = []
    all_scores = []

    if use_tiling:
        tiles, coords = create_tiles(img, tile_size=(640, 640), overlap=0.2)
        for tile, (x_off, y_off) in zip(tiles, coords):
            results = model.predict(source=tile, conf=CONF_THRESH, iou=NMS_IOU, verbose=False)[0]
            boxes = results.boxes.xyxy.cpu().clone()
            scores = results.boxes.conf.cpu().clone()
            if boxes.shape[0] > 0:
                # convert to original image coordinates
                boxes[:, [0, 2]] += x_off
                boxes[:, [1, 3]] += y_off
                all_boxes.append(boxes)
                all_scores.append(scores)
    else:
        results = model.predict(source=img, conf=CONF_THRESH, iou=NMS_IOU, verbose=False)[0]
        all_boxes.append(results.boxes.xyxy.cpu())
        all_scores.append(results.boxes.conf.cpu())

    if len(all_boxes) == 0 or all_boxes[0].shape[0] == 0:
        print("No Braille dots detected.")
        return

    all_boxes = torch.cat(all_boxes)
    all_scores = torch.cat(all_scores)

    # Apply Soft-NMS
    final_boxes, final_scores = soft_nms(all_boxes, all_scores, iou_thresh=NMS_IOU, sigma=SOFTNMS_SIGMA)

    # Draw final boxes
    for (x1, y1, x2, y2), s in zip(final_boxes, final_scores):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 1)
        cv2.putText(img, f"{s:.2f}", (int(x1), int(y1) - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

    cv2.imshow("Braille Detection", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


# ===================== Run =====================
if __name__ == "__main__":
    detect_braille(MODEL_PATH, IMAGE_PATH, use_tiling=True)


In [1]:
import cv2
import torch
from ultralytics import YOLO
import numpy as np
import os

# ===================== Config =====================
MODEL_PATH = r"D:\Papers\Braille\braille_char\v8n\weights\best.pt"
IMAGE_PATH = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\Screenshot 2025-09-05 160624.png"

CONF_THRESH = 0.05  # Lowered to allow more detections
NMS_IOU = 0.01  # Even lower to prevent suppression of overlapping dots
SOFTNMS_SIGMA = 0.5
DEBUG_MODE = True  # Set to True to save debug tiles
DEBUG_DIR = "debug_tiles"  # Directory to save debug images

# Create debug directory if needed
if DEBUG_MODE and not os.path.exists(DEBUG_DIR):
    os.makedirs(DEBUG_DIR)

# ===================== IoU Function =====================
def bbox_iou(box1, box2):
    """
    box1: (N,4), box2: (M,4)
    xyxy format: [x1, y1, x2, y2]
    """
    x1 = torch.max(box1[:, None, 0], box2[:, 0])
    y1 = torch.max(box1[:, None, 1], box2[:, 1])
    x2 = torch.min(box1[:, None, 2], box2[:, 2])
    y2 = torch.min(box1[:, None, 3], box2[:, 3])

    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])

    union = area1[:, None] + area2 - inter
    return inter / (union + 1e-6)


# ===================== Soft-NMS =====================
def soft_nms(boxes, scores, iou_thresh=0.05, sigma=0.5, score_thresh=0.001):
    boxes = boxes.clone()
    scores = scores.clone()
    N = boxes.shape[0]

    for i in range(N):
        max_pos = i + torch.argmax(scores[i:])
        # Swap highest score box to position i
        boxes[i], boxes[max_pos] = boxes[max_pos].clone(), boxes[i].clone()
        scores[i], scores[max_pos] = scores[max_pos], scores[i]

        if i < N - 1:
            ious = bbox_iou(boxes[i].unsqueeze(0), boxes[i + 1:]).squeeze(0)
            decay = torch.exp(-(ious ** 2) / sigma)
            scores[i + 1:] *= decay

    keep = scores > score_thresh
    return boxes[keep], scores[keep]


# ===================== Tiling Functions =====================
def create_tiles(img, tile_size=(640, 640), overlap=0.2):
    h, w, _ = img.shape
    th, tw = tile_size
    step_h = int(th * (1 - overlap))
    step_w = int(tw * (1 - overlap))
    tiles = []
    coords = []
    for y in range(0, h, step_h):
        for x in range(0, w, step_w):
            y1 = y
            x1 = x
            y2 = min(y + th, h)
            x2 = min(x + tw, w)
            tiles.append(img[y1:y2, x1:x2])
            coords.append((x1, y1, x2-x1, y2-y1))  # Store x, y, width, height
    return tiles, coords


def create_subtiles(tile, sub_tile_size=(160, 160), overlap=0.1):
    h, w, _ = tile.shape
    sth, stw = sub_tile_size
    step_h = int(sth * (1 - overlap))
    step_w = int(stw * (1 - overlap))
    sub_tiles = []
    coords = []
    for y in range(0, h, step_h):
        for x in range(0, w, step_w):
            y1 = y
            x1 = x
            y2 = min(y + sth, h)
            x2 = min(x + stw, w)
            sub_tiles.append(tile[y1:y2, x1:x2])
            coords.append((x1, y1, x2-x1, y2-y1))  # Store x, y, width, height
    return sub_tiles, coords


# ===================== Main Detection =====================
def detect_braille(model_path, image_path, use_tiling=True):
    model = YOLO(model_path)
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not load image from {image_path}")
        return

    all_boxes = []
    all_scores = []

    if use_tiling:
        # Create 640x640 tiles
        tiles, tile_coords = create_tiles(img, tile_size=(640, 640), overlap=0.2)
        
        for tile_idx, (tile, (x_tile, y_tile, w_tile, h_tile)) in enumerate(zip(tiles, tile_coords)):
            # Process the main tile
            results = model.predict(source=tile, conf=CONF_THRESH, iou=NMS_IOU, verbose=False)[0]
            boxes = results.boxes.xyxy.cpu().clone()
            scores = results.boxes.conf.cpu().clone()
            
            if boxes.shape[0] > 0:
                # Convert to original image coordinates
                boxes[:, [0, 2]] += x_tile
                boxes[:, [1, 3]] += y_tile
                all_boxes.append(boxes)
                all_scores.append(scores)
            
            # Save debug tile if enabled
            if DEBUG_MODE:
                debug_tile = tile.copy()
                if boxes.shape[0] > 0:
                    for (x1, y1, x2, y2), sc in zip(boxes, scores):
                        cv2.rectangle(debug_tile, (int(x1-x_tile), int(y1-y_tile)), 
                                     (int(x2-x_tile), int(y2-y_tile)), (0, 0, 255), 1)
                        cv2.putText(debug_tile, f"{sc:.2f}", (int(x1-x_tile), int(y1-y_tile)-3),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
                cv2.imwrite(f"{DEBUG_DIR}/tile_{tile_idx}_{x_tile}_{y_tile}.png", debug_tile)
            
            # Create and process 160x160 sub-tiles
            sub_tiles, sub_coords = create_subtiles(tile, sub_tile_size=(160, 160), overlap=0.1)
            
            for sub_idx, (sub_tile, (x_sub, y_sub, w_sub, h_sub)) in enumerate(zip(sub_tiles, sub_coords)):
                # Process the sub-tile
                results = model.predict(source=sub_tile, conf=CONF_THRESH, iou=NMS_IOU, verbose=False)[0]
                boxes = results.boxes.xyxy.cpu().clone()
                scores = results.boxes.conf.cpu().clone()
                
                if boxes.shape[0] > 0:
                    # Convert to tile coordinates first, then to image coordinates
                    boxes[:, [0, 2]] += x_sub + x_tile
                    boxes[:, [1, 3]] += y_sub + y_tile
                    all_boxes.append(boxes)
                    all_scores.append(scores)
                
                # Save debug sub-tile if enabled
                if DEBUG_MODE:
                    debug_sub_tile = sub_tile.copy()
                    if boxes.shape[0] > 0:
                        for (x1, y1, x2, y2), sc in zip(boxes, scores):
                            # Convert back to sub-tile coordinates for visualization
                            x1_sub = int(x1 - x_tile - x_sub)
                            y1_sub = int(y1 - y_tile - y_sub)
                            x2_sub = int(x2 - x_tile - x_sub)
                            y2_sub = int(y2 - y_tile - y_sub)
                            cv2.rectangle(debug_sub_tile, (x1_sub, y1_sub), (x2_sub, y2_sub), (255, 0, 0), 1)
                            cv2.putText(debug_sub_tile, f"{sc:.2f}", (x1_sub, y1_sub-3),
                                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)
                    cv2.imwrite(f"{DEBUG_DIR}/sub_tile_{tile_idx}_{sub_idx}_{x_tile+x_sub}_{y_tile+y_sub}.png", debug_sub_tile)
    else:
        # Process the whole image without tiling
        results = model.predict(source=img, conf=CONF_THRESH, iou=NMS_IOU, verbose=False)[0]
        all_boxes.append(results.boxes.xyxy.cpu())
        all_scores.append(results.boxes.conf.cpu())

    if len(all_boxes) == 0 or all_boxes[0].shape[0] == 0:
        print("No Braille dots detected.")
        return

    all_boxes = torch.cat(all_boxes)
    all_scores = torch.cat(all_scores)

    # Apply Soft-NMS
    final_boxes, final_scores = soft_nms(all_boxes, all_scores, iou_thresh=NMS_IOU, sigma=SOFTNMS_SIGMA)

    # Draw final boxes
    for (x1, y1, x2, y2), s in zip(final_boxes, final_scores):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(img, f"{s:.2f}", (int(x1), int(y1) - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

    # Save and display the result
    cv2.imwrite("braille_detection_result.png", img)
    cv2.imshow("Braille Detection", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()


# ===================== Run =====================
if __name__ == "__main__":
    detect_braille(MODEL_PATH, IMAGE_PATH, use_tiling=True)

In [3]:
import cv2
import torch
from ultralytics import YOLO
import numpy as np
import os

# ===================== Config =====================
MODEL_PATH = r"D:\Papers\Braille\braille_char\v8n\weights\best.pt"
IMAGE_PATH = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\Screenshot 2025-09-05 160624.png"

CONF_THRESH = 0.05  # Lowered to allow more detections
NMS_IOU = 0.01  # Even lower to prevent suppression of overlapping dots
SOFTNMS_SIGMA = 0.5
DEBUG_MODE = True
DEBUG_DIR = "debug_tiles"

if DEBUG_MODE and not os.path.exists(DEBUG_DIR):
    os.makedirs(DEBUG_DIR)

# ===================== IoU Function =====================
def bbox_iou(box1, box2):
    x1 = torch.max(box1[:, None, 0], box2[:, 0])
    y1 = torch.max(box1[:, None, 1], box2[:, 1])
    x2 = torch.min(box1[:, None, 2], box2[:, 2])
    y2 = torch.min(box1[:, None, 3], box2[:, 3])

    inter = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])
    union = area1[:, None] + area2 - inter
    return inter / (union + 1e-6)

# ===================== Soft-NMS (with class IDs) =====================
def soft_nms(boxes, scores, classes, iou_thresh=0.05, sigma=0.5, score_thresh=0.001):
    boxes = boxes.clone()
    scores = scores.clone()
    classes = classes.clone()
    N = boxes.shape[0]

    for i in range(N):
        max_pos = i + torch.argmax(scores[i:])
        # Swap highest score box to position i
        boxes[i], boxes[max_pos] = boxes[max_pos].clone(), boxes[i].clone()
        scores[i], scores[max_pos] = scores[max_pos], scores[i]
        classes[i], classes[max_pos] = classes[max_pos], classes[i]

        if i < N - 1:
            ious = bbox_iou(boxes[i].unsqueeze(0), boxes[i + 1:]).squeeze(0)
            decay = torch.exp(-(ious ** 2) / sigma)
            scores[i + 1:] *= decay

    keep = scores > score_thresh
    return boxes[keep], scores[keep], classes[keep]

# ===================== Tiling Functions =====================
def create_tiles(img, tile_size=(640, 640), overlap=0.2):
    h, w, _ = img.shape
    th, tw = tile_size
    step_h = int(th * (1 - overlap))
    step_w = int(tw * (1 - overlap))
    tiles, coords = [], []
    for y in range(0, h, step_h):
        for x in range(0, w, step_w):
            y2 = min(y + th, h)
            x2 = min(x + tw, w)
            tiles.append(img[y:y2, x:x2])
            coords.append((x, y, x2 - x, y2 - y))
    return tiles, coords

def create_subtiles(tile, sub_tile_size=(160, 160), overlap=0.1):
    h, w, _ = tile.shape
    sth, stw = sub_tile_size
    step_h = int(sth * (1 - overlap))
    step_w = int(stw * (1 - overlap))
    sub_tiles, coords = [], []
    for y in range(0, h, step_h):
        for x in range(0, w, step_w):
            y2 = min(y + sth, h)
            x2 = min(x + stw, w)
            sub_tiles.append(tile[y:y2, x:x2])
            coords.append((x, y, x2 - x, y2 - y))
    return sub_tiles, coords

# ===================== Main Detection =====================
def detect_braille(model_path, image_path, use_tiling=True):
    model = YOLO(model_path)
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not load image from {image_path}")
        return []

    all_boxes, all_scores, all_classes = [], [], []
    names = model.names  # class name dictionary

    # ---------- Run detection (same as before) ----------
    if use_tiling:
        tiles, tile_coords = create_tiles(img, (640, 640), overlap=0.2)
        for tile_idx, (tile, (x_tile, y_tile, _, _)) in enumerate(zip(tiles, tile_coords)):
            results = model.predict(source=tile, conf=CONF_THRESH, iou=NMS_IOU, verbose=False)[0]
            boxes = results.boxes.xyxy.cpu().clone()
            scores = results.boxes.conf.cpu().clone()
            classes = results.boxes.cls.cpu().clone().int()
            if boxes.shape[0] > 0:
                boxes[:, [0, 2]] += x_tile
                boxes[:, [1, 3]] += y_tile
                all_boxes.append(boxes)
                all_scores.append(scores)
                all_classes.append(classes)

            # Sub-tiles (same logic as before)
            sub_tiles, sub_coords = create_subtiles(tile, (160, 160), overlap=0.1)
            for (x_sub, y_sub, _, _), sub_tile in zip(sub_coords, sub_tiles):
                results = model.predict(source=sub_tile, conf=CONF_THRESH, iou=NMS_IOU, verbose=False)[0]
                boxes = results.boxes.xyxy.cpu().clone()
                scores = results.boxes.conf.cpu().clone()
                classes = results.boxes.cls.cpu().clone().int()
                if boxes.shape[0] > 0:
                    boxes[:, [0, 2]] += x_sub + x_tile
                    boxes[:, [1, 3]] += y_sub + y_tile
                    all_boxes.append(boxes)
                    all_scores.append(scores)
                    all_classes.append(classes)
    else:
        results = model.predict(source=img, conf=CONF_THRESH, iou=NMS_IOU, verbose=False)[0]
        all_boxes.append(results.boxes.xyxy.cpu())
        all_scores.append(results.boxes.conf.cpu())
        all_classes.append(results.boxes.cls.cpu().int())

    if len(all_boxes) == 0:
        print("No Braille characters detected.")
        return []

    # ---------- Concatenate ----------
    all_boxes = torch.cat(all_boxes)
    all_scores = torch.cat(all_scores)
    all_classes = torch.cat(all_classes)

    # ---------- Apply Soft-NMS ----------
    final_boxes, final_scores, final_classes = soft_nms(all_boxes, all_scores, all_classes,
                                                       iou_thresh=NMS_IOU, sigma=SOFTNMS_SIGMA)

    # ---------- Build results list ----------
    detections = []
    for (x1, y1, x2, y2), s, cls_id in zip(final_boxes, final_scores, final_classes):
        detections.append({
            "class": names[int(cls_id)],
            "score": float(s),
            "bbox": (float(x1), float(y1), float(x2), float(y2))
        })

    # ---------- Sort detections into reading order ----------
    # Sort by y first, then by x (with some tolerance for line grouping)
    detections = sorted(detections, key=lambda d: (d["bbox"][1], d["bbox"][0]))

    # Group into lines using y-difference threshold
    lines = []
    current_line = []
    line_threshold = 20  # adjust depending on Braille row spacing

    for det in detections:
        if not current_line:
            current_line.append(det)
        else:
            prev_y = current_line[-1]["bbox"][1]
            if abs(det["bbox"][1] - prev_y) < line_threshold:
                current_line.append(det)
            else:
                lines.append(sorted(current_line, key=lambda d: d["bbox"][0]))
                current_line = [det]
    if current_line:
        lines.append(sorted(current_line, key=lambda d: d["bbox"][0]))

    # Flatten back into ordered sequence
    ordered_classes = []
    for line in lines:
        ordered_classes.extend([d["class"] for d in line])

    # ---------- Draw final visualization ----------
    for det in detections:
        x1, y1, x2, y2 = map(int, det["bbox"])
        class_name = det["class"]
        s = det["score"]
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, f"{class_name} {s:.2f}", (x1, y1 - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    cv2.imwrite("braille_detection_result.png", img)

    return ordered_classes

# ===================== Run =====================
if __name__ == "__main__":
    detected_text = detect_braille(MODEL_PATH, IMAGE_PATH, use_tiling=True)
    print("Detected characters in order:")
    print("".join(detected_text))


Detected characters in order:
RRRRRRRTRRRNRFFRNRNFFFNRRRRNTRRRRRRTRTFNNDNRRRTNNTTTTTTTFTTFTTTFFRRRTTTTRTTRRRRRFFNRRNFFTFYNNRNNNNYYNYATDRRTNRTRRTRRRRPPRRTRNRRPRRNPTNRRNRTNTNNFTANRRRRRRRRPPTTTNRNFTNOTYNNNYHODFNNNYNYFFFDHPYFDYNLJNRNNHNNFHRYNHNYNYNYYRRYNRYRYFFRARYRYYRAYYAAYNLPIHLDRNYDNDHYHDFFNHDNNNYYAOYYYYYAEEYYYYEAYTYYHADPIHYALYYHADHAPYLLDPLYYLYYLYYAEDAYEDEYYYPYUAYYPYYPEDIJLLYYYYDYYYDYTTHEFU


In [2]:
!pip install symspellpy

  Using cached symspellpy-6.9.0-py3-none-any.whl.metadata (3.9 kB)
   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ------------------------ --------------- 1.6/2.6 MB 13.9 MB/s eta 0:00:01
   ---------------------------------------- 2.6/2.6 MB 11.6 MB/s eta 0:00:00


In [12]:
!pip install pyspellchecker

   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.2 MB 12.7 MB/s eta 0:00:01
   ---------------------------- ----------- 5.2/7.2 MB 19.9 MB/s eta 0:00:01
   ---------------------------------------- 7.2/7.2 MB 16.5 MB/s eta 0:00:00


In [ ]:
import os
import re
import csv
import cv2
import numpy as np
from datetime import datetime
from ultralytics import YOLO
from collections import defaultdict
import threading
import time

# clustering helpers
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


# Optional TTS (safe-guarded)
try:
    import pyttsx3
except Exception:
    pyttsx3 = None

# Local Transformer model for explanation
try:
    from transformers import pipeline
except Exception:
    pipeline = None

# --------------------------- Frequency Dictionary ---------------------------
class FrequencyDictionary:
    def __init__(self, dictionary_path):
        self.words = set()
        self.word_frequencies = {}
        
        try:
            with open(dictionary_path, 'r', encoding='utf-8') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        word = parts[0].lower()
                        frequency = int(parts[1])
                        self.words.add(word)
                        self.word_frequencies[word] = frequency
            print(f"Loaded {len(self.words)} words from dictionary")
        except Exception as e:
            print(f"Error loading dictionary: {e}")
            # Fallback to common words if dictionary loading fails
            self.words = set()

    def contains(self, word):
        return word.lower() in self.words

    def get_frequency(self, word):
        return self.word_frequencies.get(word.lower(), 0)

# --------------------------- Enhanced Spell Checking ---------------------------
class EnhancedSpellChecker:
    def __init__(self, dictionary_path):
        self.dictionary = FrequencyDictionary(dictionary_path)
    
    def correct_sentence(self, sentence):
        words = sentence.lower().split()
        corrected_words = []
        
        for word in words:
            # Check if word is in dictionary
            if self.dictionary.contains(word):
                corrected_words.append(word)
            else:
                # Try to split the word into dictionary words
                split_found = False
                for i in range(3, len(word)-2):
                    part1 = word[:i]
                    part2 = word[i:]
                    
                    if self.dictionary.contains(part1) and self.dictionary.contains(part2):
                        corrected_words.append(part1)
                        corrected_words.append(part2)
                        split_found = True
                        break
                
                if not split_found:
                    corrected_words.append(word)  # Keep original if no split found
        
        return " ".join(corrected_words)

# --------------------------- Line Boundary Fixer ---------------------------
def fix_line_boundaries(text: str) -> str:
    """Ensure spaces between words across line breaks."""
    # Replace newline without space → add space
    text = text.replace("\n", " ")
    # Collapse multiple spaces
    text = " ".join(text.split())
    return text

# --------------------------- Local Transformer Explainer ---------------------------
def explain_with_transformer(text: str, model=None):
    """
    Use a local transformer model to explain the detected text.
    """
    if not model:
        return "Transformer model not available"
    
    try:
        # Use the model to generate an explanation
        explanation = model(f"Explain this Braille text: {text}")[0]['generated_text']
        return explanation
    except Exception as e:
        return f"Transformer Error: {str(e)}"

# --------------------------- Cognitive Analysis ---------------------------
def generate_cognitive_explanation(text: str, model=None):
    """
    Generate a cognitive analysis of the detected text using local transformer.
    """
    if not model:
        return "Cognitive analysis not available (Transformer not configured)"
    
    try:
        # Use the model to generate cognitive analysis
        analysis = model(f"Analyze this Braille text cognitively: {text}")[0]['generated_text']
        return analysis
    except Exception as e:
        return f"Cognitive Analysis Error: {str(e)}"

# --------------------------- Braille mapping (Grade-1 basic) ---------------------------
# Mapping for 6-dot braille tuple -> lowercase letter (extend as needed)
BRAILLE_TO_CHAR = {
    # index layout (1..6):
    # (1 4)
    # (2 5)
    # (3 6)
    (1,0,0,0,0,0): 'a',
    (1,1,0,0,0,0): 'b',
    (1,0,0,1,0,0): 'c',
    (1,0,0,1,1,0): 'd',
    (1,0,0,0,1,0): 'e',
    (1,1,0,1,0,0): 'f',
    (1,1,0,1,1,0): 'g',
    (1,1,0,0,1,0): 'h',
    (0,1,0,1,0,0): 'i',
    (0,1,0,1,1,0): 'j',
    (1,0,1,0,0,0): 'k',
    (1,1,1,0,0,0): 'l',
    (1,0,1,1,0,0): 'm',
    (1,0,1,1,1,0): 'n',
    (1,0,1,0,1,0): 'o',
    (1,1,1,1,0,0): 'p',
    (1,1,1,1,1,0): 'q',
    (1,1,1,0,1,0): 'r',
    (0,1,1,1,0,0): 's',
    (0,1,1,1,1,0): 't',
    (1,0,1,0,0,1): 'u',
    (1,1,1,0,0,1): 'v',
    (0,1,0,1,1,1): 'w',
    (1,0,1,1,0,1): 'x',
    (1,0,1,1,1,1): 'y',
    (1,0,1,0,1,1): 'z',
    (0,0,0,0,0,0): ' ',  # Space
}

# --------------------------- Audio Feedback Functions ---------------------------
def play_sound(sound_type):
    """Play different sounds for confident vs uncertain letters"""
    try:
        import winsound
        if sound_type == "confident":
            winsound.Beep(1000, 100)  # High pitch for confident
        elif sound_type == "uncertain":
            winsound.Beep(500, 100)   # Low pitch for uncertain
    except:
        pass  # Sound playback not available

# --------------------------- Reading Modes Manager ---------------------------
class ReadingModeManager:
    def __init__(self, tts_engine=None):
        self.mode = "sentence"  # sentence, word, char
        self.tts_engine = tts_engine
        self.current_sentence = ""
        self.current_words = []
        self.current_word_index = 0
        self.current_char_index = 0
        self.items = []
        
    def set_content(self, sentence, items):
        self.current_sentence = sentence
        self.current_words = sentence.split()
        self.current_word_index = 0
        self.current_char_index = 0
        self.items = items
        
    def set_mode(self, mode):
        if mode in ["sentence", "word", "char"]:
            self.mode = mode
            return True
        return False
    
    def read_next(self):
        if self.mode == "sentence":
            self.read_sentence()
        elif self.mode == "word":
            self.read_word()
        elif self.mode == "char":
            self.read_char()
    
    def read_sentence(self):
        if self.tts_engine:
            self.tts_engine.say(self.current_sentence)
            self.tts_engine.runAndWait()
    
    def read_word(self):
        if self.current_word_index < len(self.current_words):
            word = self.current_words[self.current_word_index]
            if self.tts_engine:
                self.tts_engine.say(word)
                self.tts_engine.runAndWait()
            self.current_word_index += 1
    
    def read_char(self):
        if self.current_char_index < len(self.current_sentence):
            char = self.current_sentence[self.current_char_index]
            
            # Check confidence for this character if available
            conf = 0.7  # Default confidence
            for item in self.items:
                if (item.get("label") == char and 
                    self.current_char_index < len(self.current_sentence) and
                    item.get("original_index") == self.current_char_index):
                    conf = item.get("conf", 0.7)
                    break
            
            if self.tts_engine:
                self.tts_engine.say(char)
                self.tts_engine.runAndWait()
            
            # Play appropriate sound based on confidence
            if conf > 0.7:
                play_sound("confident")
            else:
                play_sound("uncertain")
            
            self.current_char_index += 1
    
    def repeat_last(self):
        if self.mode == "sentence":
            self.read_sentence()
        elif self.mode == "word" and self.current_word_index > 0:
            prev_index = self.current_word_index - 1
            if prev_index < len(self.current_words):
                word = self.current_words[prev_index]
                if self.tts_engine:
                    self.tts_engine.say(word)
                    self.tts_engine.runAndWait()
        elif self.mode == "char" and self.current_char_index > 0:
            prev_index = self.current_char_index - 1
            if prev_index < len(self.current_sentence):
                char = self.current_sentence[prev_index]
                if self.tts_engine:
                    self.tts_engine.say(char)
                    self.tts_engine.runAndWait()

# --------------------------- Enhanced Utility Functions ---------------------------
def sanitize_filename(char):
    if not char or str(char).isspace():
        return "space"
    if char == "?":
        return "unknown"
    safe = re.sub(r'[^A-Za-z0-9_-]', '', str(char))
    return safe or "unknown"

def write_csv(items, csv_path):
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["index", "label", "confidence", "x1", "y1", "x2", "y2",
                    "padded_x1", "padded_y1", "padded_x2", "padded_y2", "crop_path", "method"])
        for idx, it in enumerate(items):
            x1, y1, x2, y2 = [int(v) for v in it["box"]]
            px1, py1, px2, py2 = it.get("padded_box", (x1, y1, x2, y2))
            w.writerow([
                idx, it["label"], f"{it['conf']:.4f}",
                x1, y1, x2, y2, px1, py1, px2, py2,
                it.get("crop_path", ""), it.get("method", "")
            ])

def save_crops(img, items, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    counts = defaultdict(int)
    saved = []
    for it in items:
        x1, y1, x2, y2 = [int(v) for v in it["box"]]
        w, h = x2 - x1, y2 - y1
        padx, pady = int(0.15 * w), int(0.15 * h)
        x1n = max(0, x1 - padx)
        y1n = max(0, y1 - pady)
        x2n = min(img.shape[1], x2 + padx)
        y2n = min(img.shape[0], y2 + pady)
        crop = img[y1n:y2n, x1n:x2n]
        if crop.size == 0:
            continue
        base = sanitize_filename(it["label"])
        counts[base] += 1
        fname = f"{base}-{counts[base]:03d}.png"
        fpath = os.path.join(out_dir, fname)
        cv2.imwrite(fpath, crop)
        it["crop_path"] = fpath
        it["padded_box"] = (x1n, y1n, x2n, y2n)
        saved.append(it)
    return saved

def log_metrics(image_path, total_chars, successful, failed, total_spaces, output_dir, method):
    os.makedirs(os.path.join(output_dir, 'logs'), exist_ok=True)
    log_path = os.path.join(output_dir, f"logs/detection_log_{datetime.now().strftime('%Y%m%d')}.csv")
    file_exists = os.path.exists(log_path)
    with open(log_path, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['Timestamp', 'Image', 'Method', 'Total', 'Decoded', 'Failed', 'Spaces', 'Success Rate'])
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        success_rate = successful / total_chars if total_chars > 0 else 0
        writer.writerow([timestamp, os.path.basename(image_path), method, total_chars, successful, failed, total_spaces, f"{success_rate:.2%}"])

def draw_grid_on_image(image, grid_pattern):
    h, w = image.shape[:2]
    grid_img = image.copy()
    # vertical midline
    cv2.line(grid_img, (w // 2, 0), (w // 2, h), (0, 255, 0), 1)
    # two horizontal lines (3 rows)
    for r in range(1, 3):
        cv2.line(grid_img, (0, r * h // 3), (w, r * h // 3), (0, 255, 0), 1)
    positions = [
        (int(w * 0.25), int(h * (1/6))),  # 1
        (int(w * 0.25), int(h * (3/6))),  # 2
        (int(w * 0.25), int(h * (5/6))),  # 3
        (int(w * 0.75), int(h * (1/6))),  # 4
        (int(w * 0.75), int(h * (3/6))),  # 5
        (int(w * 0.75), int(h * (5/6))),  # 6
    ]
    dot_radius = max(2, min(w, h) // 15)
    for i, has_dot in enumerate(grid_pattern):
        if has_dot:
            cv2.circle(grid_img, positions[i], dot_radius, (0, 0, 255), -1)
    return grid_img

def visualize_gaps(img, boxes, rows, avg_width, avg_height, total_spaces, word_gaps):
    debug_img = img.copy()
    for box in boxes:
        cv2.rectangle(debug_img, (int(box[0]), int(box[1])),
                      (int(box[2]), int(box[3])), (0, 255, 0), 2)
    for i, row in enumerate(rows):
        if i > 0:
            prev_bottom = max([b[3] for b in rows[i-1]])
            current_top = min([b[1] for b in row])
            gap = current_top - prev_bottom
            cv2.line(debug_img, (0, int(prev_bottom)),
                     (debug_img.shape[1], int(prev_bottom)), (255, 0, 0), 2)
            cv2.line(debug_img, (0, int(current_top)),
                     (debug_img.shape[1], int(current_top)), (0, 0, 255), 2)
            cv2.putText(debug_img, f"Row Gap: {gap:.1f}px",
                        (10, int((prev_bottom + current_top) / 2)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    
    # Visualize word gaps
    for gap_info in word_gaps:
        gap_center_x, gap_center_y, gap = gap_info
        cv2.line(debug_img, (int(gap_center_x - gap/2), int(gap_center_y)),
                 (int(gap_center_x + gap/2), int(gap_center_y)), (0, 255, 255), 2)
        cv2.putText(debug_img, f"{gap:.1f}px",
                    (int(gap_center_x - 20), int(gap_center_y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)
    
    cv2.putText(debug_img, f"Avg Width: {avg_width:.1f}px", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(debug_img, f"Avg Height: {avg_height:.1f}px", (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(debug_img, f"Total Spaces: {total_spaces}", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    return debug_img

# --------------------------- Improved Row clustering ---------------------------
def cluster_rows(boxes, height_threshold=0.5):
    """Sort boxes into rows: top-to-bottom, left-to-right with improved clustering."""
    if len(boxes) == 0:
        return []
    
    # Calculate vertical centers
    vertical_centers = [(b[1] + b[3]) / 2.0 for b in boxes]
    avg_height = np.mean([b[3] - b[1] for b in boxes])
    
    # Handle single row case
    if len(boxes) == 1:
        return [boxes]
    
    # Use K-means clustering for better row separation
    vertical_centers = np.array(vertical_centers).reshape(-1, 1)
    
    # Try different numbers of clusters to find the best fit
    best_score = -np.inf
    best_clusters = None
    max_clusters = min(10, len(boxes))
    
    for n_clusters in range(2, max_clusters + 1):  # Start from 2 clusters
        if n_clusters > len(boxes):
            continue
            
        kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(vertical_centers)
        
        # Only calculate silhouette score if we have multiple clusters
        if n_clusters > 1:
            try:
                score = silhouette_score(vertical_centers, kmeans.labels_)
            except:
                score = -1  # Assign low score if silhouette fails
        else:
            score = -1
        
        if score > best_score:
            best_score = score
            best_clusters = kmeans.labels_
    
    # If no clusters found with n>=2, use single cluster
    if best_clusters is None:
        return [boxes]
    
    # Group boxes by cluster
    rows = [[] for _ in range(len(set(best_clusters)))]
    for i, cluster_id in enumerate(best_clusters):
        rows[cluster_id].append(boxes[i])
    
    # Sort rows by average y-position (top to bottom)
    row_centers = [np.mean([(b[1] + b[3]) / 2 for b in row]) for row in rows]
    sorted_indices = np.argsort(row_centers)
    sorted_rows = [rows[i] for i in sorted_indices]
    
    # Sort each row by x-position (left to right)
    for i, row in enumerate(sorted_rows):
        sorted_rows[i] = sorted(row, key=lambda b: b[0])
    
    return sorted_rows

# --------------------------- Simple crop decoder (2x3 grid) ---------------------------
def enhanced_decode_braille_from_crop(crop_img, yolo_label=None, yolo_confidence=0.0):
    """
    Lightweight 2x3 grid analyzer:
    - binarize crop
    - split into 6 ROIs
    - mark a dot if foreground ratio > threshold
    - map pattern to BRAILLE_TO_CHAR; fallback to yolo_label
    """
    if crop_img is None or crop_img.size == 0:
        return (yolo_label or '?'), [0,0,0,0,0,0], float(yolo_confidence or 0.0)

    gray = cv2.cvtColor(crop_img, cv2.COLOR_BGR2GRAY)
    # normalize & binarize
    gray = cv2.equalizeHist(gray)
    blur = cv2.GaussianBlur(gray, (3,3), 0)
    th = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                               cv2.THRESH_BINARY_INV, 15, 5)

    # Resize to a canonical size to split evenly
    H, W = 120, 80  # 3 rows, 2 cols
    th_resized = cv2.resize(th, (W, H), interpolation=cv2.INTER_NEAREST)

    # ROIs: 2 columns x 3 rows
    grid = [0]*6
    dot_scores = []
    col_w = W // 2
    row_h = H // 3
    idx = 0
    for c in range(2):
        for r in range(3):
            x1 = c * col_w
            y1 = r * row_h
            roi = th_resized[y1:y1+row_h, x1:x1+col_w]
            ratio = float(np.count_nonzero(roi)) / float(roi.size)
            grid[idx] = 1 if ratio > 0.12 else 0  # threshold can be tuned
            dot_scores.append(ratio)
            idx += 1

    pattern = tuple(grid)
    decoded = BRAILLE_TO_CHAR.get(pattern, None)

    # Confidence: combine YOLO conf and grid contrast
    grid_strength = float(np.mean(dot_scores))
    if decoded is None:
        decoded = yolo_label if yolo_label is not None else '?'
        conf = max(float(yolo_confidence), grid_strength)
    else:
        conf = 0.5 * float(yolo_confidence) + 0.5 * grid_strength

    return decoded, grid, conf

# --------------------------- Item builders ---------------------------
def yolo_only_decoding(model, img, boxes, cls_ids, confs, names):
    items = []
    for i, box in enumerate(boxes):
        label = names[cls_ids[i]] if names else str(cls_ids[i])
        conf = float(confs[i])
        items.append({
            "box": box.tolist() if hasattr(box, "tolist") else list(box),
            "label": label,
            "conf": conf,
            "method": "yolo_only"
        })
    return items

def rpn_grid_decoding(model, img, boxes, cls_ids, confs, names):
    items = []
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = [int(v) for v in box]
        w, h = x2 - x1, y2 - y1
        padx, pady = int(0.15 * w), int(0.15 * h)
        x1n = max(0, x1 - padx)
        y1n = max(0, y1 - pady)
        x2n = min(img.shape[1], x2 + padx)
        y2n = min(img.shape[0], y2 + pady)
        crop_img = img[y1n:y2n, x1n:x2n]
        if crop_img.size == 0:
            continue
        yolo_label = names[cls_ids[i]] if names else str(cls_ids[i])
        yolo_conf = float(confs[i])
        decoded, grid, confidence = enhanced_decode_braille_from_crop(
            crop_img, yolo_label=yolo_label, yolo_confidence=yolo_conf
        )
        items.append({
            "box": box.tolist() if hasattr(box, "tolist") else list(box),
            "label": decoded,
            "conf": float(confidence),
            "grid": grid,
            "method": "rpn_grid",
            "yolo_label": yolo_label,
            "yolo_conf": yolo_conf
        })
    return items

# --------------------------- Improved Assignment helpers ---------------------------
def assign_rows_to_items(items, rows):
    """
    Assign a row_id to each item by IoU overlap with row boxes.
    rows: list of list of boxes
    """
    def iou(a, b):
        ax1, ay1, ax2, ay2 = a
        bx1, by1, bx2, by2 = b
        inter_x1, inter_y1 = max(ax1, bx1), max(ay1, by1)
        inter_x2, inter_y2 = min(ax2, bx2), min(ay2, by2)
        iw, ih = max(0, inter_x2 - inter_x1), max(0, inter_y2 - inter_y1)
        inter = iw * ih
        area_a = (ax2 - ax1) * (ay2 - ay1)
        area_b = (bx2 - bx1) * (by2 - by1)
        union = area_a + area_b - inter + 1e-6
        return inter / union

    for it in items:
        it["row_id"] = -1

    for row_id, row in enumerate(rows):
        for it in items:
            if it["row_id"] != -1:
                continue
            # choose the best overlap with any box in this row
            best = 0.0
            for rb in row:
                best = max(best, iou(it["box"], rb))
            if best > 0.2:  # overlap threshold
                it["row_id"] = row_id

    # any leftover -> assign by nearest row center in Y
    if rows:
        row_centers = [np.mean([(b[1]+b[3])/2.0 for b in r]) for r in rows]
        for it in items:
            if it["row_id"] == -1:
                cy = (it["box"][1] + it["box"][3]) / 2.0
                nearest = int(np.argmin([abs(cy - rc) for rc in row_centers]))
                it["row_id"] = nearest

    return items

# --------------------------- Spell Checking and Word Segmentation ---------------------------
def apply_spell_check(sentence, spell_checker):
    """Apply spell checking to fix common spacing and word issues"""
    if spell_checker is None:
        return sentence
    
    return spell_checker.correct_sentence(sentence)

def detect_word_boundaries(items, rows, horizontal_space_threshold=1.5):
    """Detect word boundaries based on spacing between characters"""
    # Group items by row
    grouped = defaultdict(list)
    for it in items:
        grouped[it["row_id"]].append(it)
    
    # Sort items in each row by x-coordinate
    for row_id in grouped:
        grouped[row_id] = sorted(grouped[row_id], key=lambda x: x["box"][0])
    
    # Calculate average character width
    if items:
        avg_width = np.mean([it["box"][2] - it["box"][0] for it in items])
    else:
        avg_width = 0
    
    # Process each row to detect word boundaries
    word_gaps = []
    for row_id, row_items in grouped.items():
        for i in range(1, len(row_items)):
            prev_box = row_items[i-1]["box"]
            curr_box = row_items[i]["box"]
            
            # Calculate gap between characters
            gap = curr_box[0] - prev_box[2]
            gap_center_x = (prev_box[2] + curr_box[0]) / 2
            gap_center_y = (prev_box[1] + prev_box[3]) / 2
            
            # If gap is significantly larger than average character width, it's a word boundary
            if gap > horizontal_space_threshold * avg_width:
                row_items[i]["word_start"] = True
                word_gaps.append((gap_center_x, gap_center_y, gap))
    
    return items, word_gaps

# --------------------------- Main processing ---------------------------
def process_braille_image(
    model,
    image_path=None,
    method="yolo_only",   # 'yolo_only' or 'rpn_grid'
    horizontal_space_threshold=1.5,  # Increased from 1.0 to 1.5 for better word separation
    vertical_space_threshold=0.8,
    newline_threshold=1.5,
    save_outputs=True,
    enable_tts=False,
    reading_mode="sentence",  # 'sentence', 'word', 'char'
    enable_spell_check=True,
    transformer_model=None,      # Local transformer model
    enable_transformer_explanation=False,  # Enable transformer explanation
    enable_cognitive_analysis=False,  # Enable cognitive analysis
    dictionary_path=None  # Path to frequency dictionary
):
    # Initialize spell checker if enabled
    spell_checker = EnhancedSpellChecker(dictionary_path) if enable_spell_check and dictionary_path else None
    
    # Initialize TTS engine if enabled
    tts_engine = None
    if enable_tts and pyttsx3 is not None:
        try:
            tts_engine = pyttsx3.init()
            tts_engine.setProperty('rate', 120)
            tts_engine.setProperty('volume', 1.0)
        except Exception as e:
            print(f"TTS Error: {e}")
            tts_engine = None
    
    # Initialize reading mode manager
    reading_manager = ReadingModeManager(tts_engine)
    reading_manager.set_mode(reading_mode)
    
    img = cv2.imread(image_path)
    if img is None:
        print(f"Image not found: {image_path}")
        return "", []

    max_dim = min(max(img.shape[:2]), 1920)
    try:
        stride = int(model.stride.max().item())
    except Exception:
        stride = 32
    imgsz = int(np.ceil(max_dim / stride) * stride)

    results = model.predict(img, imgsz=imgsz, verbose=False)
    r0 = results[0]

    boxes = r0.boxes.xyxy.cpu().numpy()
    cls_ids = r0.boxes.cls.cpu().numpy().astype(int)
    confs = r0.boxes.conf.cpu().numpy() if hasattr(r0.boxes, "conf") else np.ones(len(cls_ids))
    names = model.names if hasattr(model, "names") else None

    # Sort boxes by position (left to right, top to bottom)
    if len(boxes) > 0:
        # Calculate center points
        centers = [(box[0] + box[2]) / 2 for box in boxes], [(box[1] + box[3]) / 2 for box in boxes]
        
        # Sort by y-position first (rows), then by x-position (within rows)
        sorted_indices = np.lexsort((centers[0], centers[1]))
        boxes = boxes[sorted_indices]
        cls_ids = cls_ids[sorted_indices]
        confs = confs[sorted_indices] if hasattr(confs, '__getitem__') else confs

    if len(boxes) > 0:
        widths = [b[2] - b[0] for b in boxes]
        heights = [b[3] - b[1] for b in boxes]
        avg_width = float(np.median(widths))
        avg_height = float(np.median(heights))
    else:
        avg_width = avg_height = 0.0

    if method == "yolo_only":
        items = yolo_only_decoding(model, img, boxes, cls_ids, confs, names)
    elif method == "rpn_grid":
        items = rpn_grid_decoding(model, img, boxes, cls_ids, confs, names)
    else:
        raise ValueError("Method must be 'yolo_only' or 'rpn_grid'")

    # Cluster rows from raw boxes and assign to items
    rows = cluster_rows(list(boxes))
    assign_rows_to_items(items, rows)
    
    # Detect word boundaries
    items, word_gaps = detect_word_boundaries(items, rows, horizontal_space_threshold)

    # Group by row_id and assemble sentence with adaptive spacing
    grouped = defaultdict(list)
    for it in items:
        grouped[it["row_id"]].append(it)

    sentence_parts = []
    total_spaces = 0

    for row_id in sorted(grouped.keys()):
        row_items = sorted(grouped[row_id], key=lambda it: it["box"][0])
        row_text = []
        for i, it in enumerate(row_items):
            if i > 0:
                prev_box = row_items[i-1]["box"]
                box = it["box"]
                gap = box[0] - prev_box[2]
                char_width = ((prev_box[2] - prev_box[0]) + (box[2] - box[0])) / 2.0
                adaptive_threshold = horizontal_space_threshold * char_width
                if gap > adaptive_threshold or it.get("word_start", False):
                    row_text.append(" ")
                    total_spaces += 1
            row_text.append(it["label"])
        row_str = "".join(row_text)

        if row_id > 0:
            current_top = min([it["box"][1] for it in row_items])
            prev_bottom = max([it["box"][3] for it in grouped[row_id - 1]])
            gap_rows = current_top - prev_bottom
            line_gap_threshold = vertical_space_threshold * avg_height
            newline_gap_threshold = newline_threshold * avg_height
            if gap_rows > newline_gap_threshold:
                sentence_parts.append("\n")
            elif gap_rows > line_gap_threshold:
                sentence_parts.append(" ")
                total_spaces += 1

        sentence_parts.append(row_str)

    raw_sentence = "".join(sentence_parts)
    
    # Fix line boundaries (ONEWHO\nLOSTHIS → ONE WHO LOST HIS)
    raw_sentence = fix_line_boundaries(raw_sentence)
    
    # Apply spell checking if enabled
    if enable_spell_check:
        corrected_sentence = apply_spell_check(raw_sentence, spell_checker)
    else:
        corrected_sentence = raw_sentence
    
    # Get transformer explanation if enabled
    transformer_explanation = ""
    if enable_transformer_explanation and transformer_model:
        transformer_explanation = explain_with_transformer(corrected_sentence, transformer_model)
    
    # Get cognitive analysis if enabled
    cognitive_analysis = ""
    if enable_cognitive_analysis and transformer_model:
        cognitive_analysis = generate_cognitive_explanation(corrected_sentence, transformer_model)
    
    # Store original indices for character-by-character reading
    char_index = 0
    for row_id in sorted(grouped.keys()):
        row_items = sorted(grouped[row_id], key=lambda it: it["box"][0])
        for it in row_items:
            it["original_index"] = char_index
            char_index += 1

    # Visualizations
    annotated_img = r0.plot()
    for it in items:
        x1, y1, x2, y2 = [int(v) for v in it["box"]]
        label = it["label"]
        cv2.putText(annotated_img, f"{label} ({it['conf']:.2f})",
                    (x1, max(10, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (50, 200, 255), 2)

    gap_img = visualize_gaps(img, boxes, rows, avg_width, avg_height, total_spaces, word_gaps)

    if save_outputs:
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_dir = f"detected_characters_{method}_{ts}"
        os.makedirs(out_dir, exist_ok=True)

        saved_items = save_crops(img, items, os.path.join(out_dir, "crops"))
        write_csv(saved_items, os.path.join(out_dir, "mapping.csv"))

        cv2.imwrite(os.path.join(out_dir, "annotated_yolo.png"), annotated_img)
        cv2.imwrite(os.path.join(out_dir, "gap_visualization.png"), gap_img)

        with open(os.path.join(out_dir, "decoded_sentence.txt"), "w", encoding="utf-8") as f:
            f.write("Raw Sentence:\n")
            f.write(raw_sentence)
            f.write("\n\nCorrected Sentence:\n")
            f.write(corrected_sentence)
            if transformer_explanation:
                f.write("\n\nTransformer Explanation:\n")
                f.write(transformer_explanation)
            if cognitive_analysis:
                f.write("\n\nCognitive Analysis:\n")
                f.write(cognitive_analysis)

        if method == "rpn_grid":
            grid_dir = os.path.join(out_dir, "grid_visualizations")
            os.makedirs(grid_dir, exist_ok=True)
            for it in saved_items:
                if "crop_path" in it and "grid" in it:
                    char_img = cv2.imread(it["crop_path"])
                    if char_img is not None:
                        grid_img = draw_grid_on_image(char_img, it["grid"])
                        base_name = os.path.splitext(os.path.basename(it["crop_path"]))[0]
                        grid_path = os.path.join(grid_dir, f"{base_name}_grid.png")
                        cv2.imwrite(grid_path, grid_img)

        successful_decodes = sum(1 for it in items if it["label"] != '?')
        failed = len(items) - successful_decodes
        log_metrics(image_path, len(items), successful_decodes, failed, total_spaces, out_dir, method)
        print(f"\nDebugging files saved to: {out_dir}")

    # Show windows
    cv2.imshow(f"Annotated ({method})", annotated_img)
    cv2.imshow("Gap Visualization", gap_img)
    print("\n" + "="*50)
    print(f"RAW SENTENCE:\n{raw_sentence}")
    print(f"CORRECTED SENTENCE:\n{corrected_sentence}")
    print(f"TOTAL SPACES: {total_spaces}")
    if transformer_explanation:
        print(f"TRANSFORMER EXPLANATION:\n{transformer_explanation}")
    if cognitive_analysis:
        print(f"COGNITIVE ANALYSIS:\n{cognitive_analysis}")
    print("="*50)
    
    # Set up reading manager
    reading_manager.set_content(corrected_sentence, items)
    
    # Start interactive mode
    print("\nInteractive Controls:")
    print("R - Repeat last")
    print("W - Spell last word")
    print("C - Spell last character")
    print("S - Switch reading mode (sentence/word/char)")
    print("E - Explain with Transformer (if enabled)")
    print("A - Cognitive Analysis (if enabled)")
    print("Q - Quit")
    print("Space - Read next (in word/char mode)")
    
    # Interactive loop
    while True:
        key = cv2.waitKey(0) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('r'):
            reading_manager.repeat_last()
        elif key == ord('w'):
            # Spell the last word
            if reading_manager.current_word_index > 0:
                word = reading_manager.current_words[reading_manager.current_word_index - 1]
                if tts_engine:
                    for char in word:
                        tts_engine.say(char)
                        tts_engine.runAndWait()
                        time.sleep(0.1)
        elif key == ord('c'):
            # Spell the last character
            if reading_manager.current_char_index > 0:
                char = corrected_sentence[reading_manager.current_char_index - 1]
                if tts_engine:
                    tts_engine.say(char)
                    tts_engine.runAndWait()
        elif key == ord('s'):
            # Switch reading mode
            if reading_manager.mode == "sentence":
                reading_manager.set_mode("word")
                print("Mode switched to: WORD")
            elif reading_manager.mode == "word":
                reading_manager.set_mode("char")
                print("Mode switched to: CHARACTER")
            else:
                reading_manager.set_mode("sentence")
                print("Mode switched to: SENTENCE")
        elif key == ord('e') and enable_transformer_explanation:
            # Explain with Transformer
            if transformer_explanation and tts_engine:
                tts_engine.say(transformer_explanation)
                tts_engine.runAndWait()
            elif not transformer_explanation and transformer_model:
                # Generate explanation on demand
                explanation = explain_with_transformer(corrected_sentence, transformer_model)
                if tts_engine:
                    tts_engine.say(explanation)
                    tts_engine.runAndWait()
            else:
                print("Transformer explanation not available")
        elif key == ord('a') and enable_cognitive_analysis:
            # Cognitive analysis
            if cognitive_analysis and tts_engine:
                tts_engine.say(cognitive_analysis)
                tts_engine.runAndWait()
            elif not cognitive_analysis and transformer_model:
                # Generate cognitive analysis on demand
                analysis = generate_cognitive_explanation(corrected_sentence, transformer_model)
                if tts_engine:
                    tts_engine.say(analysis)
                    tts_engine.runAndWait()
            else:
                print("Cognitive analysis not available")
        elif key == 32:  # Space bar
            reading_manager.read_next()
    
    cv2.destroyAllWindows()

    # Optional TTS on the decoded sentence
    if enable_tts and tts_engine is not None:
        try:
            tts_engine.say(corrected_sentence)
            tts_engine.runAndWait()
        except Exception as e:
            print(f"TTS Error: {e}")

    return corrected_sentence, items, total_spaces

# --------------------------- Comparison runner ---------------------------
def compare_methods(model, image_path, dictionary_path):
    yolo_sentence, yolo_items, yolo_spaces = process_braille_image(
        model, image_path, method="yolo_only", save_outputs=False, dictionary_path=dictionary_path
    )
    rpn_sentence, rpn_items, rpn_spaces = process_braille_image(
        model, image_path, method="rpn_grid", save_outputs=False, dictionary_path=dictionary_path
    )

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    comp_dir = f"comparison_{ts}"
    os.makedirs(comp_dir, exist_ok=True)

    img = cv2.imread(image_path)
    comp_img = img.copy() if img is not None else None

    if comp_img is not None:
        # Attempt pairwise overlay by index (best-effort)
        for yolo_it, rpn_it in zip(yolo_items, rpn_items):
            x1, y1, x2, y2 = [int(v) for v in yolo_it["box"]]
            cv2.putText(comp_img, f"Y:{yolo_it['label']}", (x1, y1 - 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            cv2.putText(comp_img, f"G:{rpn_it['label']}", (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)
        cv2.imwrite(os.path.join(comp_dir, "comparison.png"), comp_img)

    with open(os.path.join(comp_dir, "comparison_report.txt"), "w", encoding="utf-8") as f:
        f.write("BRAILLE RECOGNITION METHODS COMPARISON\n")
        f.write("======================================\n\n")
        f.write(f"Image: {os.path.basename(image_path)}\n")
        f.write(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

        f.write("YOLO-ONLY METHOD:\n")
        f.write("-----------------\n")
        f.write(f"Sentence: {yolo_sentence}\n")
        f.write(f"Spaces: {yolo_spaces}\n")
        yolo_success = sum(1 for it in yolo_items if it["label"] != '?')
        f.write(f"Success Rate: {yolo_success}/{len(yolo_items)} ({(yolo_success/len(yolo_items)*100 if yolo_items else 0):.1f}%)\n\n")

        f.write("RPN+GRID METHOD:\n")
        f.write("----------------\n")
        f.write(f"Sentence: {rpn_sentence}\n")
        f.write(f"Spaces: {rpn_spaces}\n")
        rpn_success = sum(1 for it in rpn_items if it["label"] != '?')
        f.write(f"Success Rate: {rpn_success}/{len(rpn_items)} ({(rpn_success/len(rpn_items)*100 if rpn_items else 0):.1f}%)\n\n")

        f.write("DIFFERENCES:\n")
        f.write("------------\n")
        if yolo_sentence == rpn_sentence:
            f.write("Both methods produced identical results.\n")
        else:
            f.write("Methods produced different results:\n")
            f.write(f"YOLO-only: {yolo_sentence}\n")
            f.write(f"RPN+Grid:  {rpn_sentence}\n")

    print(f"\nComparison report saved to: {comp_dir}")
    return yolo_sentence, rpn_sentence, comp_dir

# --------------------------- Run ---------------------------
if __name__ == "__main__":
    # Update these paths
    model_path = r"D:\Papers\Braille\braille_char\v8n\weights\best.pt"
    image_path = r"C:\Users\mogno\OneDrive\Pictures\Screenshots 1\Screenshot 2025-09-05 160624.png"
    dictionary_path = r"C:\Users\mogno\Downloads\frequency_dictionary_en_82_765.txt"
    
    # Load local transformer model if available
    transformer_model = None
    if pipeline is not None:
        try:
            transformer_model = pipeline('text-generation', model='gpt2')
        except Exception as e:
            print(f"Error loading transformer model: {e}")
            transformer_model = None

    model = YOLO(model_path)

    # Choose: "yolo_only", "rpn_grid", or "comparison"
    mode = "rpn_grid"

    if mode == "yolo_only":
        process_braille_image(
            model,
            image_path=image_path,
            method="yolo_only",
            save_outputs=True,
            enable_tts=False,
            reading_mode="sentence",
            enable_spell_check=True,
            transformer_model=transformer_model,
            enable_transformer_explanation=False,  # Set to True if you want transformer explanations
            enable_cognitive_analysis=False,  # Set to True if you want cognitive analysis
            dictionary_path=dictionary_path
        )
    elif mode == "rpn_grid":
        process_braille_image(
            model,
            image_path=image_path,
            method="rpn_grid",
            save_outputs=True,
            enable_tts=False,
            reading_mode="sentence",
            enable_spell_check=True,
            transformer_model=transformer_model,
            enable_transformer_explanation=False,  # Set to True if you want transformer explanations
            enable_cognitive_analysis=False,  # Set to True if you want cognitive analysis
            dictionary_path=dictionary_path
        )
    elif mode == "comparison":
        yolo_sentence, rpn_sentence, comp_dir = compare_methods(model, image_path, dictionary_path)
        print("\n" + "="*60)
        print("COMPARISON RESULTS:")
        print("="*60)
        print(f"YOLO-only: {yolo_sentence}")
        print(f"RPN+Grid:  {rpn_sentence}")
        print("="*60)

Device set to use cpu


Loaded 82834 words from dictionary

Debugging files saved to: detected_characters_rpn_grid_20250909_135753

RAW SENTENCE:
gbthbgbhqUthgbtrjlhggtrrUNgbtqrbYhgwjthgrjhqgtthhYhbg VjllbghthhYY hqtrjlhgtO OthhYqrbYhgggYjqthhh grhhqthhh Ygrjh tYhhbbYYUtr hY hbqqYhtr h hh
CORRECTED SENTENCE:
gbthbgbhquthgbtrjlhggtrrungbtqrbyhgwjthgrjhqgtthhyhbg vjllbghthhyy hqtrjlhgto othhyqrbyhgggyjqthhh grhhqthhh ygrjh tyhhbbyyutr hy hbqqyhtr h hh
TOTAL SPACES: 8

Interactive Controls:
R - Repeat last
W - Spell last word
C - Spell last character
S - Switch reading mode (sentence/word/char)
E - Explain with Transformer (if enabled)
A - Cognitive Analysis (if enabled)
Q - Quit
Space - Read next (in word/char mode)


In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ===================== Load Image =====================
image_path = r"C:\Users\mogno\Project\yolo_grid_detection_20250905_120421\crops\Y_013_crop.png"
img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

# ===================== Preprocess =====================
# Denoise
img_blur = cv2.medianBlur(img, 5)

# Adaptive threshold
thresh = cv2.adaptiveThreshold(
    img_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 11, 2
)

# Ensure dots are white, background black
white_ratio = np.sum(thresh == 255) / thresh.size
if white_ratio > 0.5:
    thresh = cv2.bitwise_not(thresh)

# ===================== Remove grid lines =====================
# Detect vertical lines
kernel_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 15))
grid_v = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_v)

# Detect horizontal lines
kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 1))
grid_h = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_h)

# Combine grid masks
grid_lines = cv2.bitwise_or(grid_v, grid_h)

# Subtract grids from threshold to keep only dots
dots_only = cv2.subtract(thresh, grid_lines)

# Cleanup small gaps
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
dots_only = cv2.morphologyEx(dots_only, cv2.MORPH_CLOSE, kernel)

# ===================== Blob Detection =====================
params = cv2.SimpleBlobDetector_Params()
params.filterByArea = True
params.minArea = 5
params.maxArea = 1000
params.filterByCircularity = True
params.minCircularity = 0.4
params.filterByConvexity = False
params.filterByInertia = False

detector = cv2.SimpleBlobDetector_create(params)
keypoints = detector.detect(dots_only)

# Fallback if nothing found
if len(keypoints) == 0:
    print("⚠️ No blobs found → fallback to connected components")
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(dots_only, connectivity=8)
    keypoints = []
    for i in range(1, n_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if 3 < area < 500:
            x, y = int(centroids[i][0]), int(centroids[i][1])
            size = int(np.sqrt(area))
            kp = cv2.KeyPoint(x, y, size)
            keypoints.append(kp)

print(f"✅ Found {len(keypoints)} dot(s)")

# ===================== Heatmap =====================
heatmap = np.zeros_like(img, dtype=np.float32)

for kp in keypoints:
    x, y = int(kp.pt[0]), int(kp.pt[1])
    size = int(max(4, kp.size // 2))
    cv2.circle(heatmap, (x, y), size, 1, -1)

# Smooth + normalize
heatmap = cv2.GaussianBlur(heatmap, (21, 21), 0)
heatmap = cv2.normalize(heatmap, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

# Overlay heatmap
overlay = cv2.addWeighted(cv2.cvtColor(img, cv2.COLOR_GRAY2BGR), 0.6, heatmap_color, 0.4, 0)

# ===================== Add 2x3 Grid =====================
rows, cols = 3, 2
h, w = img.shape[:2]
grid_img = overlay.copy()
for i in range(1, rows):
    y = int(i * h / rows)
    cv2.line(grid_img, (0, y), (w, y), (255, 255, 255), 1)
for j in range(1, cols):
    x = int(j * w / cols)
    cv2.line(grid_img, (x, 0), (x, h), (255, 255, 255), 1)

# ===================== Show Results =====================
plt.figure(figsize=(18, 6))

plt.subplot(1, 5, 1)
plt.title("Original")
plt.imshow(img, cmap="gray")
plt.axis("off")

plt.subplot(1, 5, 2)
plt.title("Thresholded")
plt.imshow(thresh, cmap="gray")
plt.axis("off")

plt.subplot(1, 5, 3)
plt.title("Dots Only")
plt.imshow(dots_only, cmap="gray")
plt.axis("off")

plt.subplot(1, 5, 4)
plt.title("Heatmap")
plt.imshow(heatmap_color)
plt.axis("off")

plt.subplot(1, 5, 5)
plt.title("Overlay + Grid")
plt.imshow(cv2.cvtColor(grid_img, cv2.COLOR_BGR2RGB))
plt.axis("off")

plt.tight_layout()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\mogno\\Project\\yolo_grid_detection_20250905_120421\\crops\\Y_013_crop.png'